# 第10章：集約とグループ操作（`groupby/agg/transform/apply`実務パターン）

In [ ]:
import pandas as pd
from pathlib import Path
DATA = Path.cwd() / "assets"
sales = pd.read_csv(DATA / "sales_sample.csv")
sales.head()

## 10.1 groupby基礎

In [ ]:
# 店舗ごとの売上合計
store_sum = sales.groupby('store')['amount'].sum().sort_values(ascending=False)
store_sum.head()

In [ ]:
# store×categoryの2軸集計
sc = sales.groupby(['store','category'])['amount'].sum().unstack('category')
sc

## 10.2 agg（複数関数/複数列）

In [ ]:
agg_multi = (sales.groupby('store')
                .agg(sum_amount=('amount','sum'),
                     mean_amount=('amount','mean'),
                     orders=('product','count'),
                     mean_qty=('qty','mean'))
               )
agg_multi

## 10.3 transform（グループ基準での列変換）

In [ ]:
# 例：カテゴリ平均でamountの欠損を補完
sales_copy = sales.copy()
sales_copy['amount_filled'] = sales_copy['amount'].fillna(
    sales_copy.groupby('category')['amount'].transform('mean')
)
sales_copy[['category','amount','amount_filled']].head(10)

In [ ]:
# 例：カテゴリ内標準化（Z-score）
sales_copy['z_in_cat'] = (sales_copy['amount'] - sales_copy.groupby('category')['amount'].transform('mean')) /                          sales_copy.groupby('category')['amount'].transform('std')
sales_copy[['category','amount','z_in_cat']].head()

## 10.4 apply（最後の手段）と代替案

In [ ]:
# applyでやる例（遅くなることが多い）
def high_value_rate(g):
    return pd.Series({'high_rate': (g['amount']>5000).mean()})
r1 = sales.groupby(['store','category']).apply(high_value_rate)
r1.head()

In [ ]:
# ベクトル化代替（推奨）
r2 = (sales.assign(high=(sales['amount']>5000))
           .groupby(['store','category'])['high']
           .mean()
           .to_frame('high_rate'))
r2.head()

## 小課題：顧客KPI（平均単価、購入頻度、直近日）

In [ ]:
# ヒント：groupby customer_id → aggでfrequency, avg_amount, last_date
sales['date'] = pd.to_datetime(sales['date'], errors='coerce')
kpi = (sales
       .groupby('customer_id')
       .agg(orders=('product','count'),
            days_active=('date', lambda s: (s.max() - s.min()).days if s.notna().any() else pd.NA),
            avg_amount=('amount','mean'),
            last_purchase=('date','max'))
      )
kpi.head()